In [ ]:
import os
print(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

In [ ]:
!ls -R /kaggle
%pip install -q pandas numpy torch transformers datasets scikit-learn tqdm matplotlib

In [59]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


# Local Jupyter Setup

If you run this notebook on a local Jupyter environment instead of Kaggle, install the required packages first.

Recommended minimal packages:
- `pandas`, `numpy`
- `torch`
- `transformers`
- `datasets`
- `scikit-learn`
- `tqdm`
- `matplotlib`

> In Jupyter, `%pip` is usually better than `!pip` because it installs into the active kernel environment.


# Load IMDB Movie Reviews

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import sklearn


,review,sentiment
18885,I am insulted and angry over the idea that a s...,negative


# Load IMDB 50K Dataset in Jupyter

If the Kaggle CSV path does not exist, the notebook can automatically download the IMDB dataset from Hugging Face via the `datasets` library.


In [61]:
# Installing Transformers library
# !pip install transformers 

In [ ]:
from pathlib import Path

def load_imdb_dataframe(local_csv_path=None, save_downloaded_csv=False):
    """
    Load IMDB 50K reviews as a pandas DataFrame with columns:
    - review
    - sentiment  ('positive' / 'negative')
    """
    if local_csv_path is None:
        candidate_paths = [
            Path('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'),
            Path('./IMDB Dataset.csv'),
            Path('./data/IMDB Dataset.csv'),
        ]
    else:
        candidate_paths = [Path(local_csv_path)]

    for path in candidate_paths:
        if path.exists():
            print(f'Loading local CSV: {path}')
            return pd.read_csv(path)

    print('Local CSV not found. Downloading IMDB dataset from Hugging Face...')
    from datasets import load_dataset

    ds = load_dataset('imdb')
    train_df = ds['train'].to_pandas()
    test_df = ds['test'].to_pandas()
    df = pd.concat([train_df, test_df], ignore_index=True)

    df['sentiment'] = df['label'].map({1: 'positive', 0: 'negative'})
    df = df[['text', 'sentiment']].rename(columns={'text': 'review'})

    if save_downloaded_csv:
        output_path = Path('./IMDB Dataset.csv')
        df.to_csv(output_path, index=False)
        print(f'Saved downloaded dataset to: {output_path.resolve()}')

    return df

df = load_imdb_dataframe(save_downloaded_csv=False)
df.sample(5)

# Load DistilBERT and Tokenizer
## DistilBERT

DistilBERT is a lightweight transformer-based language model distilled from BERT. 
It retains most of BERT’s language understanding ability while being smaller and faster. 
In this project, DistilBERT is used as the text encoder to generate contextual embeddings from input reviews.

## Tokenizer

The tokenizer converts raw text into a sequence of tokens that the model can process. 
It maps words or subwords into token IDs and applies padding or truncation so that the input can be fed into the DistilBERT model.

In [62]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import InputExample
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Data Preprocessing
## For training model with BERT, we need to do some additional Prepriocessing: 
## **raw dataframe => inputexample => ReviewDataset(tokenizaition + tensor) => DataLoader(batch + shuffle) => DistilBERT**

### 1. cat2num:
Map the sentiment column into 0/1



In [63]:
# changing positive and negative into numeric values

def cat2num(value):
    if value=='positive': 
        return 1
    else: 
        return 0
    
df['sentiment']  =  df['sentiment'].apply(cat2num)
train = df[:45000]
test = df[45000:]

### 1. tokenize: text => token
### 2. convert_tokens_to_ids: token => vocab ids

In [64]:
# But first see BERT tokenizer exmaples and other required stuff!

example='In this Kaggle notebook, I will do sentiment analysis using BERT with Huggingface'
tokens=tokenizer.tokenize(example)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(tokens)
print(token_ids)

['in', 'this', 'ka', '##ggle', 'notebook', ',', 'i', 'will', 'do', 'sentiment', 'analysis', 'using', 'bert', 'with', 'hugging', '##face']
[1999, 2023, 10556, 24679, 14960, 1010, 1045, 2097, 2079, 15792, 4106, 2478, 14324, 2007, 17662, 12172]



* The original word has been split into smaller subwords and characters. This is because DistilBert Vocabulary is fixed with a size of ~30K tokens. Words that are not part of vocabulary are represented as subwords and characters. For example, 'ka', '##ggle' or 'hugging', '##face'.

* Tokenizer takes the input sentence and will decide to keep every word as a whole word, split it into sub words(with special representation of first sub-word and subsequent subwords — see ## symbol in the example above) or as a last resort decompose the word into individual characters. Because of this, we can always represent a word as, at the very least, the collection of its individual characters.

Reference: https://medium.com/@dhartidhami/understanding-bert-word-embeddings-7dc4d2ea54ca

### 1. convert_data_to_examples:
This will accept our train and test datasets and convert each row into an InputExample object. Transformer only accepts example objects.

### 2. ReviewDataset: 
This function will tokenize the InputExample objects, then create the required input format with the tokenized objects, finally, create an input dataset that we can feed to the model.


In [65]:
def convert_data_to_examples(train, test, review, sentiment): 
    train_InputExamples = train.apply(lambda x: InputExample(guid=None, # Globally unique ID for bookkeeping, unused in this case
                                                          text_a = x[review], 
                                                          label = x[sentiment]), axis = 1)

    validation_InputExamples = test.apply(lambda x: InputExample(guid=None, # Globally unique ID for bookkeeping, unused in this case
                                                          text_a = x[review], 
                                                          label = x[sentiment]), axis = 1,)
  
    return train_InputExamples, validation_InputExamples

train_InputExamples, validation_InputExamples = convert_data_to_examples(train,  test, 'review',  'sentiment')
                                                                         

In [66]:
train_InputExamples[1]

InputExample(guid=None, text_a='A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating eve

## ReviewDataset Class

The `ReviewDataset` class prepares the text data so it can be used by the model.

### 1. Tokenization

The tokenizer converts text into model inputs:

- `input_ids` – token IDs used by the model  
- `attention_mask` – indicates which tokens are real tokens vs padding  
- `token_type_ids` – identifies sentence segments (used in some transformer models)

Special tokens used by BERT:

- `[CLS]` – added at the beginning for classification tasks  
- `[SEP]` – marks the end of a sentence  
- `[PAD]` – used for padding shorter sequences  
- `[UNK]` – represents unknown tokens not in the vocabulary

### 2. Padding and Truncation

All sequences are padded or truncated to a fixed length (`max_length = 128`).

### 3. Convert to Tensor

Tokenized outputs are converted into PyTorch tensors.

### 4. Store as Features

The processed inputs are stored in a feature list so they can be accessed by the dataset during training.

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=128):
        self.features = []

        for e in tqdm(examples):
            encoded = tokenizer(
                e.text_a,
                add_special_tokens=True,         # [CLS], [SEP]
                max_length=max_length,
                padding='max_length',            # pad to max_length
                truncation=True,
                return_attention_mask=True,
                return_token_type_ids=True,
            )

            self.features.append({
                "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
                "token_type_ids": torch.tensor(encoded["token_type_ids"], dtype=torch.long),
                "labels": torch.tensor(e.label, dtype=torch.long)
            })

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

train_dataset = ReviewDataset(train_InputExamples, tokenizer, max_length=128)
val_dataset = ReviewDataset(validation_InputExamples, tokenizer, max_length=128)

### 1. DataLoader
Generate batch data for DistilBERT

In [68]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Model Architecture (Training)

In [69]:
import torch.nn as nn
from transformers import DistilBertModel

class DistilBERTMeanPoolClassifier(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_labels=2, dropout=0.2):
        super().__init__()
        
        # DistilBERT backbone
        self.backbone = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size  # usually 768
        
        # 2-layer classifier head
        self.attention = nn.Linear(hidden_size, 1)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def mean_pool(self, last_hidden_state, attention_mask):
        """
        last_hidden_state: (B, T, H)
        attention_mask:    (B, T)
        """
        mask = attention_mask.unsqueeze(-1).float()   # (B, T, 1)
        masked_hidden = last_hidden_state * mask      # pad tokens become 0
        summed = masked_hidden.sum(dim=1)             # (B, H)
        counts = mask.sum(dim=1).clamp(min=1e-9)      # (B, 1)
        mean_pooled = summed / counts                 # (B, H)
        return mean_pooled
    
    def attention_pool(self, last_hidden_state, attention_mask):

        # (B,T,H) -> (B,T,1)
        scores = self.attention(last_hidden_state)

        # remove padding influence
        scores = scores.masked_fill(
            attention_mask.unsqueeze(-1) == 0,
            -1e9
        )

        # normalize
        weights = torch.softmax(scores, dim=1)

        # weighted sum
        pooled = torch.sum(weights * last_hidden_state, dim=1)

        return pooled
    
    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # token-level outputs: (B, T, H)
        last_hidden_state = outputs.last_hidden_state
        
        # mean pooling
        # pooled = self.mean_pool(last_hidden_state, attention_mask)
        pooled = self.attention_pool(last_hidden_state, attention_mask)
        
        # classifier head
        logits = self.classifier(pooled)
        return logits
    

# model = DistilBERTMeanPoolClassifier(
#     model_name="distilbert-base-uncased",
#     num_labels=2,
#     dropout=0.2
# )

In [70]:
# model.summary()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DistilBERTMeanPoolClassifier(
    model_name="distilbert-base-uncased",
    num_labels=2,
    dropout=0.2
).to(device)

print(model)
criterion = nn.CrossEntropyLoss()

### 1. Training Stage

In [72]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if step % 100 == 0:
            acc = correct / total
            print(f"Step {step}, Loss: {loss.item():.4f}, Accuracy: {acc:.4f}")

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

### 1. Eval Stage

In [73]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for step, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            if step % 100 == 0:
                acc = correct / total
                print(f"Step {step}, Loss: {loss.item():.4f}, Accuracy: {acc:.4f}")

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# Freeze DistilBERT backbone
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

stage1_epochs = 3
stage1_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(stage1_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    stage1_history["train_loss"].append(train_loss)
    stage1_history["train_acc"].append(train_acc)
    stage1_history["val_loss"].append(val_loss)
    stage1_history["val_acc"].append(val_acc)

    print(f"[Stage 1] Epoch {epoch+1}/{stage1_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

In [ ]:
# Unfreeze all parameters
for param in model.backbone.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

stage2_epochs = 3
stage2_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(stage2_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    stage2_history["train_loss"].append(train_loss)
    stage2_history["train_acc"].append(train_acc)
    stage2_history["val_loss"].append(val_loss)
    stage2_history["val_acc"].append(val_acc)

    print(f"[Stage 2] Epoch {epoch+1}/{stage2_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
print("Validation Accuracy:", accuracy)

f1 = f1_score(all_labels, all_preds)
print("Validation F1 score:", f1)

roc_auc = roc_auc_score(all_labels, all_probs)
print("Validation ROC-AUC:", roc_auc)

# Evaluation Metrics

1. **Confusion matrix**  
2. **F1 score**  
3. **Training vs. validation loss / accuracy curves**  
4. **Example predictions**  
5. **Model settings**  


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

print(classification_report(all_labels, all_preds, target_names=['negative', 'positive']))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix:")
print(cm)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest')
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=['negative', 'positive'],
    yticklabels=['negative', 'positive'],
    ylabel='True label',
    xlabel='Predicted label',
    title='Confusion Matrix'
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

threshold = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > threshold else "black"
        )

fig.tight_layout()
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc_value = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_value:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

# Plot training curves for both stages
stage1_epoch_range = range(1, len(stage1_history["train_loss"]) + 1)
stage2_epoch_range = range(1, len(stage2_history["train_loss"]) + 1)

plt.figure(figsize=(6, 5))
plt.plot(stage1_epoch_range, stage1_history["train_loss"], marker='o', label='Stage 1 Train Loss')
plt.plot(stage1_epoch_range, stage1_history["val_loss"], marker='o', label='Stage 1 Val Loss')
plt.plot(stage2_epoch_range, stage2_history["train_loss"], marker='s', label='Stage 2 Train Loss')
plt.plot(stage2_epoch_range, stage2_history["val_loss"], marker='s', label='Stage 2 Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves: Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(stage1_epoch_range, stage1_history["train_acc"], marker='o', label='Stage 1 Train Accuracy')
plt.plot(stage1_epoch_range, stage1_history["val_acc"], marker='o', label='Stage 1 Val Accuracy')
plt.plot(stage2_epoch_range, stage2_history["train_acc"], marker='s', label='Stage 2 Train Accuracy')
plt.plot(stage2_epoch_range, stage2_history["val_acc"], marker='s', label='Stage 2 Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Curves: Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Demo Inference Function

Use the functions below to let a user input a sentence or review and get a positive / negative prediction from the trained model.


In [ ]:
import torch.nn.functional as F

def predict_sentiment(text, model, tokenizer, device, max_length=128):
    model.eval()

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    pred_idx = int(probs.argmax())
    label_map = {0: 'negative', 1: 'positive'}

    return {
        'label': label_map[pred_idx],
        'negative_probability': float(probs[0]),
        'positive_probability': float(probs[1]),
    }

def demo_sentiment_prediction():
    user_text = input('Enter a movie review: ')
    result = predict_sentiment(user_text, model, tokenizer, device)
    print('\nPrediction result:')
    print(result)


# For interactive demo in Jupyter:
demo_sentiment_prediction()